# Patent analysis

In [1]:
import pandas as pd

In [2]:
# AltairSaver = altair_save_utils.AltairSaver()

In [3]:
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import utils
import importlib
importlib.reload(utils);

2024-07-05 10:28:59,063 - botocore.credentials - INFO - Found credentials in environment variables.
2024-07-05 10:29:00,323 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [72]:
# Labelled data
data_df = utils.load_patents_data().query("topics != 'arts'")

In [73]:
len(data_df.drop_duplicates("id"))

25972

In [74]:
# Taxonomy dataframe
topics_df = utils.load_topic_data()

In [75]:
topics_df

,topic,type,subtype,name
0,genetics,Biosciences,Genetics,Genetics
1,neuroscience,Biosciences,Neuroscience,Neuroscience
2,operations,Child care & preschool,Operations,Operations
3,preschool,Child care & preschool,Preschool,Preschool
4,cognitive,Development & learning,Cognitive development,Cognitive development
5,communication,Development & learning,Communication and language,Communication and language
6,arts,Development & learning,Expressive arts and design,Expressive arts and design
7,literacy,Development & learning,Literacy,Literacy
8,mathematics,Development & learning,Mathematics,Mathematics
9,emotional,Development & learning,Personal social emotional,Personal social emotional


In [76]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

## Corrections

In [77]:
check_types = ['Child care & preschool', 'Development & learning', 'Health', 'Parenting', 'Society']

tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

type_ids = data_exploded_df.query("type in @check_types").id.to_list()

# Data not tagged with the main application types
_data_df = (
    data_df
    .query("id in @tech_ids and ~(id in @type_ids)")
)

# include also patents mentioning health or medical
adjust_ids = _data_df[_data_df.text.str.contains('health|medical', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', health'

# include also patents mentioning education
adjust_ids = _data_df[_data_df.text.str.contains('education', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', cognitive'

# include also patents mentioning education
adjust_ids = _data_df[_data_df.text.str.contains('parenting|parent', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', parenting2'

# preschool
adjust_ids = _data_df[_data_df.text.str.contains('kindergarten', case=False)].id.to_list()
data_df.loc[data_df.id.isin(adjust_ids), 'topics'] = data_df.loc[data_df.id.isin(adjust_ids), 'topics'] + ', preschool'

In [78]:
adjust_ids = data_df[data_df.text.str.contains('internet|online', case=False)].id.to_list()
remove_ids = data_df[data_df.text.str.contains('internet of things', case=False)].id.to_list()
data_df.query("id in @adjust_ids and ~(id in @remove_ids)").sample(10)

,id,text,dataset,topics,year,country_code
9181,CN-108447565-A,A small-for-gestational-age infant disease pre...,patents,"health, ai2, infancy",2018,CN
16087,CN-105187470-A,An intelligent kindergarten care system based ...,patents,NaN,2015,CN
12459,CN-110909752-A,A multipurpose network platform for individual...,patents,infancy,2020,CN
23396,CN-109035743-A,A kind of home intelligence tele-control syste...,patents,"robotics, ai2, mobile, infancy",2018,CN
13824,CN-204595856-U,A kind of intelligent baby bed Telemedicine Sy...,patents,"ai2, mobile, sleep, infancy",2015,CN
11408,CN-219983481-U,An online steam direct injection sterilizer. T...,patents,NaN,2023,CN
2414,KR-20170117857-A,Communication type system using wheel switch. ...,patents,infancy,2017,KR
14026,TW-M576445-U,Infant physiological monitoring device. The pr...,patents,infancy,2019,TW
21641,CN-210294143-U,Online body fluid detection device and online ...,patents,"oral, ai2, infancy",2020,CN
3648,CN-108606545-A,A kind of remote controlled infanette. The pre...,patents,"robotics, mobile, sleep, infancy",2018,CN


In [79]:
# data_df = data_df.query("country_code != 'CN'")

In [80]:
len(data_df)

25972

In [81]:
from discovery_child_development import PROJECT_DIR, S3_BUCKET
from nesta_ds_utils.loading_saving import S3

In [82]:
data_df.to_csv(PROJECT_DIR / 'outputs/data/data_patents.csv', index=False)
S3.upload_obj(
    data_df,
    bucket = S3_BUCKET,
    path_to = '2024-07-iss-child-development/outputs/data/data_patents.csv'
)

In [59]:
# Transform to one id and topic pair per row
data_exploded_df = utils.explode_data(data_df).query("topic != 'arts'")

### Patents with not application areas

In [60]:
type_ids = data_exploded_df.query("type in @check_types").id.to_list()

# Data not tagged with the main application types
_data_df = (
    data_df
    .query("id in @tech_ids and ~(id in @type_ids)")
)

In [61]:
_data_df.sample(20)

,id,text,dataset,topics,year,country_code
20880,EP-3164990-A4,Systems and methods for configuring baby monit...,patents,"ai2, infancy",2017,EP
18384,US-2022151397-A1,Wearable baby carriers with multiple oparation...,patents,wearables,2022,US
10534,CN-218044489-U,baby care equipment. The utility model disclos...,patents,"ai2, mobile, infancy",2022,CN
19594,CN-211529356-U,Intelligent learning machine for infants. The ...,patents,"ai2, robotics, infancy",2020,CN
15224,CN-206777457-U,diaper with induction chip. The diaper with in...,patents,"mobile, infancy",2017,CN
7372,CN-107577203-A,"Intelligent stroller control method, device an...",patents,"ai2, infancy",2018,CN
13961,CN-214906737-U,Visual function detection auxiliary system. Th...,patents,"ai2, neuroscience, infancy",2021,CN
755,KR-20170084868-A,Smart robot-based context-aware infants soluti...,patents,"ai2, robotics, infancy",2017,KR
26006,WO-2018053045-A2,Breast sense feeding monitor. The Breast Sense...,patents,"wearables, infancy",2018,WO
4563,US-2022048520-A1,Vehicle and control method for vehicle. A vehi...,patents,"robotics, infancy",2022,US


For example: Monitoring and snesor systems (without clear reason for monitoring), intelligent devices (eg, ceiling fan, anti-theft baby clothes), behaviour detection and recognition, general infant care systems, baby vomit monitor etc

## Baseline trends

Baseline trends for patent counts

In [83]:
importlib.reload(utils);
baseline_df = utils.get_baseline_patents()

In [86]:
baseline_df.to_csv(PROJECT_DIR / 'outputs/data/baseline_data_patents.csv', index=False)
S3.upload_obj(
    baseline_df,
    bucket = S3_BUCKET,
    path_to = '2024-07-iss-child-development/outputs/data/baseline_data_patents.csv'
)

In [63]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,7572871.2,31.192759


In [64]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(counts = lambda df: df.counts/1e+6),
    ["Total"],
    variable= "counts",
    variable_title = "Publications (millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [65]:
ts_counts = utils.get_timeseries(data_df.drop_duplicates('id'), column='id')

In [66]:
ts_counts

,year,counts
0,2013,1046
1,2014,1301
2,2015,2056
3,2016,2107
4,2017,2288
5,2018,2802
6,2019,2937
7,2020,3065
8,2021,3509
9,2022,2751


In [67]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,2874.4,4.273078


In [68]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total"),
    ["Total"],
    variable= "counts",
    variable_title = "Publications",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [71]:
utils.get_data_distribution(
    data_exploded_df.query("year >= 2019 and year <= 2023"),
    column='type', values=['id'])

,type,counts,counts_prop
0,Biosciences,275,0.019
1,Child care & preschool,619,0.043
2,Development & learning,767,0.053
3,General,10587,0.737
4,Health,3490,0.243
5,Parenting,277,0.019
6,Society,31,0.002
7,Technology,2618,0.182


In [346]:
importlib.reload(utils);
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='id')

,magnitude,growth,type,counts
6,6.2,175.000000,Society,31
1,123.8,28.771930,Child care & preschool,619
2,153.4,25.360231,Development & learning,767
0,55.0,20.833333,Biosciences,275
4,698.0,9.443861,Health,3490
3,2117.4,6.734238,General,10587
7,523.6,-5.220126,Technology,2618
5,55.4,-25.136612,Parenting,277


In [347]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "counts",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [29]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile'}

In [30]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
)

In [31]:
# ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_counts_tech, 'counts')

alt.Chart(...)

In [32]:
au.ts_magnitude_growth_(ts_counts_tech, year_start = 2019, year_end = 2023)

,magnitude,growth
counts,523.6,-5.220126


### Distribution of different technologies

In [33]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [34]:
# Total tech funding
counts_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").id.nunique()

In [35]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
    .assign(counts_prop = lambda df: df.counts/counts_total)
)

tech_subtype_dist

,subtype,counts,counts_prop
0,AI,1637,0.625286
1,Immersive tech,874,0.333843
2,Internet,10,0.003820
3,Mobile,677,0.258594


### Growth of technology topics

In [36]:
column = 'subtype'
value = 'counts'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,327.4,9.630459,AI
0,174.8,-14.874552,Immersive tech
0,2.0,-33.333333,Internet
0,135.4,-42.229730,Mobile


In [37]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile"],
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [38]:
tech_ids = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    .query("type == 'Technology'")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

In [39]:
len(tech_ids_5y)

2618

### Application distribution

In [40]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology' and type != 'General'"),
    column=column, 
    values=['id'],
    ts=True
)



In [41]:
tech_applications_df

,type,counts,counts_prop
0,Biosciences,85,0.032
1,Child care & preschool,173,0.066
2,Development & learning,260,0.099
3,General,2055,0.785
4,Health,920,0.351
5,Parenting,261,0.1
6,Society,2,0.001
7,Technology,2618,1.0


In [42]:
fig = pu.ts_smooth(
    tech_applications_ts,
    tech_applications_ts[column].unique(),
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [43]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')

,magnitude,growth,type,counts
7,0.4,inf,Society,2
6,17.0,177.272727,Biosciences,85
3,184.0,5.482042,Health,920
1,52.0,0.714286,Development & learning,260
2,411.0,-0.165289,General,2055
5,523.6,-5.220126,Technology,2618
0,34.6,-23.622047,Child care & preschool,173
4,52.2,-28.491620,Parenting,261


In [44]:
trends_df = utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
(
    tech_applications_df
    .merge(trends_df.drop('counts', axis=1), on='type')[['type', 'magnitude', 'growth', 'counts', 'counts_prop']]
    .query("type != 'Technology' and type != 'General'")
)

,type,magnitude,growth,counts,counts_prop
0,Biosciences,17.0,177.272727,85,0.032
1,Child care & preschool,34.6,-23.622047,173,0.066
2,Development & learning,52.0,0.714286,260,0.099
4,Health,184.0,5.482042,920,0.351
5,Parenting,52.2,-28.491620,261,0.1
6,Society,0.4,inf,2,0.001


In [45]:
from discovery_child_development.utils import chart_trends
importlib.reload(chart_trends);

chart_trends.estimate_trend_type(
    trends_df.query("type != 'Technology' and type != 'General'"), 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,magnitude,growth,type,counts,trend_type_suggestion
7,0.4,inf,Society,2,emerging
6,17.0,177.272727,Biosciences,85,emerging
3,184.0,5.482042,Health,920,hot
1,52.0,0.714286,Development & learning,260,hot
0,34.6,-23.622047,Child care & preschool,173,dormant
4,52.2,-28.491620,Parenting,261,stable


In [46]:
# scatter chart of trends_df
import altair as alt
alt.Chart(
    trends_df.query("type != 'Technology' and type != 'General'")
).mark_point().encode(
    x='magnitude:Q',
    y='growth:Q',
    color='type:N',
    tooltip=['type', 'magnitude', 'growth']
)


alt.Chart(...)

In [178]:
# pd.set_option('display.max_colwidth', 200)
# (
#     data_exploded_df
#     .query('id in @tech_ids')
#     # .query("subtype == 'Personal social emotional'")
#     .query("type == 'Social'")
#     .drop_duplicates(['id'])
#     .sort_values('year', ascending=False)
# )[['id', 'text', 'topics', 'year']]

### Application distribution: More granular subtypes

In [310]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('counts', ascending=False)

,subtype,counts,counts_prop,type
8,Infancy,1979,0.756,General
24,Sleep,430,0.164,Health
6,Health,321,0.123,Health
23,Preschool,162,0.062,Child care & preschool
21,Physical development,161,0.061,Health
2,Cognitive development,158,0.06,Development & learning
4,Games,97,0.037,General
14,Neuroscience,84,0.032,Biosciences
22,Prenatal,43,0.016,Health
26,Special educational needs,37,0.014,Development & learning


In [123]:
(
    utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='id')
    .sort_values(['type', 'growth'], ascending=False)
    # .sort_values('growth', ascending=False)
)

,magnitude,growth,subtype,counts,type
12,327.4,9.630459,AI,1637,Technology
19,174.8,-14.874552,Immersive tech,874,Technology
22,2.0,-33.333333,Internet,10,Technology
24,135.4,-42.229730,Mobile,677,Technology
0,0.4,inf,Social services,2,Social
9,1.0,50.000000,Parenting,5,Parenting
5,2.4,125.000000,Nutrition & weight,12,Health
8,8.6,75.000000,Prenatal,43,Health
10,32.2,17.283951,Physical development,161,Health
11,24.2,16.666667,Health,121,Health


In [37]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [38]:
fig = pu.ts_smooth(
    tech_applications_ts,
    cats,
    variable= "counts",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

### Which technology is applied to most to subtype X?

## Insight 3: Geographical insights

- Top countries in terms of counts
- UK vs baseline growth for overall counts, in technology counts and application counts

In [205]:
# data_exploded_df.explode('country_code').drop_duplicates(['id', 'country_code']).isnull().sum()
# Consider lack of information

In [437]:
data_countries_df = (
    data_exploded_df
    # .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id'])    
)
country_codes = data_countries_df.country_code.unique()

growth_df = []
ts_counts = []
for country_code in country_codes:
    country_df = data_countries_df.query("country_code == @country_code")
    _ts_counts = utils.get_timeseries(country_df, column='id').assign(country_code = country_code)
    growth_df.append(
        au.ts_magnitude_growth_(
            ts_df = _ts_counts,
            year_start = 2019,
            year_end = 2023  
        )
        .assign(country_code = country_code)
        .reset_index(drop=True)
    )
    ts_counts.append(_ts_counts)
growth_df = pd.concat(growth_df, ignore_index=True)
ts_counts = pd.concat(ts_counts, ignore_index=True)

In [439]:
(
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(20)
)

,magnitude,growth,country_code
0,334.6,-15.964126,CN
1,80.0,32.596685,KR
2,36.6,-13.274336,US
6,20.8,-9.836066,WO
11,12.2,11.428571,JP
4,7.8,141.666667,EP
5,6.4,69.230769,TW
3,4.2,800.000000,AU
9,2.8,120.000000,TR
7,2.8,125.000000,CA


In [442]:
data_df.sort_values('year')

,id,text,dataset,topics,year,country_code
25840,KR-200468095-Y1,"Early Childhood Learning Diocese. The present invention relates to a learning teaching aid for children, through the activity of observing blocks stacked in a stacking space from various direction...",patents,"physical, infancy",2013,KR
11811,WO-2013185134-A2,"Blood biomarkers for necrotizing enterocolitis. Necrotizing Enterocolitis (NEC) biomarkers, NEC biomarker panels, and methods for obtaining a NEC signature for a sample are provided. Also provided...",patents,infancy,2013,WO
13660,WO-2013179239-A1,Smart green baby bottle assembly. A smart green baby bottle assembly comprising: a hollow container portion comprising an opening surrounded by screw threads; at least one baby feeding nipple; and...,patents,NaN,2013,WO
25273,US-8387785-B2,Two chambered baby bottle attachment for storing and dispensing infant formula. An infant formula storage and dispensing assembly that attaches to a baby bottle. The assembly is made up of two par...,patents,infancy,2013,US
11816,CN-203041506-U,"A smart baby cradle. An intelligent baby cradle, including a sleeping basket, the two sides of the sleeping basket are respectively fixed on the shaft sleeves through suspenders, the shaft sleeves...",patents,"sleep, infancy",2013,CN
...,...,...,...,...,...,...
7536,JP-7217662-B2,childcare equipment with seat.,patents,NaN,2023,JP
24724,EP-4153185-A1,Use of human milk oligosaccharides in nutritional compositions for enhancing bone development and/or bone strength. The invention relates to the use of composition comprising sialylated and fucosy...,patents,"nutrition, infancy",2023,EP
3659,CN-218922598-U,"An infant EEG electrode wire and its fixing device. The utility model relates to the technical field of medical supplies, and discloses an electroencephalogram electrode wire for infants and a fix...",patents,infancy,2023,CN
10651,AU-2017232122-B2,Apparatus for use in a child safety seat. An apparatus for a child safety seat is disclosed. The apparatus comprises a releasable \n5 connector configured to engage with an anchoring point provide...,patents,NaN,2023,AU


In [440]:
countries = ['US', 'GB', 'CN', 'KR', 'JP']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "counts",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

In [42]:
data_countries_df = (
    data_exploded_df
    .explode('country_code')
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id', 'subtype']) 
    .query("country_code == 'GB'")   
)

In [43]:
data_countries_df.groupby('subtype').agg(counts=('id', 'nunique')).reset_index()

,subtype,counts
0,AI,9
1,Immersive tech,3
2,Mobile,10


## Check detailed applications

In [150]:
importlib.reload(utils);
df = utils.get_counts_by_application(data_exploded_df, topics_df)
df.to_csv(utils.PROJECT_DIR / 'outputs/data/tables/tech_applications_patents.csv', index=False)

In [151]:
df.Total.sum()

1144

## Export tables

In [14]:
def check_if_nan_only_list_elements(x):
    return all([pd.isna(i) for i in x])

def convert_topic_columns(df):
    for col in ['topic', 'minor_category', 'major_category', 'topic_code']:
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: list(x) if check_if_nan_only_list_elements(x)==False else [])})
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: ", ".join([xx for xx in x if isinstance(xx, str)]) if len(x)>0 else "")})
    return df

In [15]:
_export_df = (
    data_df
    .fillna({'topics': ''})
    .assign(topics = lambda df: df['topics'].apply(lambda x: [t.strip() for t in x.split(',') if isinstance(t, str)]))
    .explode('topics')
    .merge(topics_df, left_on='topics', right_on='topic', how='left')
    .rename(columns={'topic': 'topic_code', 'type': 'major_category', 'subtype': 'minor_category', 'name': 'topic'})
    .drop('topics', axis=1)
)  
export_df = (
    data_df[['id', 'text', 'dataset', 'year', 'country_code']]
    .merge(
        _export_df[['id', 'topic', 'major_category', 'minor_category', 'topic_code']].groupby('id').agg(set),
        on='id',
        how='left'
    )
    .pipe(convert_topic_columns)
    .assign(url = lambda df: "https://patents.google.com/patent/" + df['id'].str.replace("-", ""))
)

In [16]:
export_df.to_csv(utils.PROJECT_DIR / "outputs/data/tables/patents_final.csv", index=False)

In [18]:
data_exploded_df.query("topic == 'parenting2'")

,id,text,dataset,topics,year,country_code,topic,type,subtype,name
141,CN-106137149-A,Newborn care system based on 4G network. This ...,patents,parenting2,2016,CN,parenting2,Parenting,Parenting,Parenting
245,KR-20170058600-A,Alarm for condition based on IoT. The present ...,patents,parenting2,2017,KR,parenting2,Parenting,Parenting,Parenting
269,KR-20220054099-A,Baby Care System. The present invention relate...,patents,parenting2,2022,KR,parenting2,Parenting,Parenting,Parenting
349,CN-109846152-A,A watch with a mother-infant parenting APP. Th...,patents,parenting2,2019,CN,parenting2,Parenting,Parenting,Parenting
390,CN-110142776-A,An early education robot for tracking and cari...,patents,parenting2,2019,CN,parenting2,Parenting,Parenting,Parenting
...,...,...,...,...,...,...,...,...,...,...
45157,KR-102308448-B1,Apparatus for Infant Management and Driving Me...,patents,parenting2,2021,KR,parenting2,Parenting,Parenting,Parenting
45219,CN-111361475-A,Child safety seat base and child safety seat. ...,patents,parenting2,2020,CN,parenting2,Parenting,Parenting,Parenting
45325,DE-202023105250-U1,A system for a multifunctional educational and...,patents,parenting2,2023,DE,parenting2,Parenting,Parenting,Parenting
45398,CN-104966426-A,A Kind of Arithmetic Learning Machine for Chil...,patents,parenting2,2015,CN,parenting2,Parenting,Parenting,Parenting
